# Briefing
Esse notebook tem como objetivo treinar um modelo de classificação de eventos sonoros. Utilizou o dataset **UrbanSound8K**, pré-separado em 10 folds, com 10 categorias de eventos sonoros.

Utilizou-se um Multilayer Perceptron para classificar o áudio com base no seu MFCC, um tipo de espectrograma específico capaz de comprimir a informação tempo-frequência em poucos `bins` espectrais.

# Importando tudo

In [12]:
# !!Panic Button
# Deletar checkpoints
## !rm -rf checkpoints
## !mkdir checkpoints
# print('done')


# import os
# os._exit(0)


done


In [1]:
from IPython.display import clear_output
# !pip install --upgrade ipython
# %load_ext autoreload
# %autoreload 2

!git clone https://github.com/y7kko/COM804-Classifier.git ykk_repo
!git -C ./ykk_repo/ pull
!pip install git+https://github.com/soundata/soundata.git
!pip install torch-audiomentations
!pip install git+https://github.com/iver56/audiomentations.git


clear_output()
import sys
import os

sys.path.append('./ykk_repo/')
os.environ['YKK_NOCHDIR'] = '1'
os.environ['US8K_PATH'] = './data/urbansound8k'

import ykk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import optuna
ykk.urban8k(root_path=os.environ['US8K_PATH']).download_dataset()
import time
import copy
from torchaudio import transforms
from pathlib import Path
Path('./checkpoints/').mkdir(parents=True, exist_ok=True)
print('Done, check console for more details')

Done, check console for more details


In [7]:
# !git -C ./ykk_repo/ log

# Declarando

## Dataset importer

In [2]:
def get_data(val_fold,fs,transform,return_dataloader=True,cache_validation=True):
    """Splits dataset"""
    train_fld = [f for f in range(1, 11) if f != val_fold]
    test_fld = train_fld[0]
    train_fld = train_fld[1:]
    
    valid_data = ykk.urban8k(augmentation= False, fs=fs, filter_folds = val_fold)
    test_data  = ykk.urban8k(augmentation= False, fs=fs, filter_folds = test_fld)
    train_data = ykk.urban8k(augmentation= True,  fs=fs, filter_folds = train_fld)

    # Aplica a mesma técnica de feature extraction para 
    # todos os subconjuntos
    for _data in [train_data,valid_data,test_data]:
            _data.set_transform(transform)

    # audiomentations não possui parâmetro de random_state, mas chama internamente o np.random.(...)
    # portanto, forçarei o random state global
    np.random.seed(0)

    train_data.cache_signals()
    train_data.cache_features(augmentation_sets=4)
    
    if cache_validation:
        valid_data.cache_features()

    if return_dataloader:
        train_dl = DataLoader(train_data, shuffle=False, batch_size= 32)
        valid_dl = DataLoader(valid_data, batch_size= 32)
        test_dl = DataLoader(test_data, batch_size= 32)
        return train_dl,valid_dl,test_dl
    else:
        return train_data,valid_data,test_data

## Parâmetros
Foi utilizado o MFCC transform para realizar extração das features. Embora alguns estudos apontem melhores resultados com STFT ou log-mel, o modelo foi treinado na CPU, e a MFCC oferece dados com menos dimensões.

- `fs`: Frequência de amostragem do sinal
- `mel_kw`: Parâmetros do espectrograma
- `n_mfcc`: número de componentes de frequência por frame de tempo

Nota: para simplificar, o tamanho da input layer foi calculado realizando a extração de features, mas há forma analítica de calcular.

In [3]:
fs = 22050

# _dB = transforms.AmplitudeToDB(stype="power", top_db=80)
# _specgram = transforms.MelSpectrogram(fs,n_fft=512,hop_length=512,n_mels=60,)
# transform = lambda x: _dB(_specgram(x))
transform = nn.Sequential(
    transforms.MelSpectrogram(fs,n_fft=512,hop_length=512,n_mels=60,),
    transforms.AmplitudeToDB(stype="power", top_db=80)
)
_d = ykk.urban8k(fs=fs)
_d.set_transform(transform)
input_shape = list((_d[0][0]).shape)
print(f'input shape: {input_shape} ({np.prod(input_shape)})')

input shape: [1, 60, 130] (7800)


## O modelo
Pode-se interpretar o MFCC como uma imagem, portanto utilizou-se uma CNN 2D. 

In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self,dropout=.5,dense1=64,dense2=32):
        super().__init__()
        # Nx-|K|+1 = 20 - 5
        # Ny-|K|+1 = 173 - 5
        self.sequence = nn.Sequential(
            # Layer1 conv
            nn.LazyConv2d(24, kernel_size=5),   # (B, 16, 16, 83)
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),      # !(B, 16, 8, 41)
            
            # Layer2 conv
            nn.LazyConv2d(32, kernel_size=3), # (B, 32, 6, 39)
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                  # !(B, 32, 3, 19)
        
            
            # Layer 3 Lin
            nn.Flatten(),                      # (B, 1824)
            nn.LazyLinear(dense1),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),

            # Layer 4 lin
            nn.Linear(dense1, dense2),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),

            # Layer 5
            nn.Linear(dense2, 10),
        )
    def forward(self, x):
        x = self.sequence(x)
        return x

# n_params = sum(p.numel() for p in NeuralNetwork().parameters())
# print(f'Number o parameters {n_params/1E3}E3')


## Trainer


In [5]:
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
class Trainer():
    def __init__(self,model,loss_fn,optimizer,valid_loss_delta=1E-4):
        self.device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
        print(f'device: {self.device}')
        self.model = model
        self.optimizer =optimizer
        self.loss_fn = loss_fn
        self.valid_loss, self.train_loss = [], []
        self.accuracy, self.f1_score = [], []
        self.epoch = 0
        self.early_stop_flag = False
        self.best_accuracy = 0
        self.best_valid_loss = 0
        self.best_epoch = 0
        self.test_stats = {}


    def train(self,train_data:DataLoader):
        size = len(train_data.dataset)
        self.model = self.model.to(self.device)
        self.model.train()
        running_loss,seen = 0.0, 0
        for batch, (X, y) in enumerate(tqdm(train_data, desc="Treino")):
            X, y = self._dev_encode(X,y)
            # Compute prediction error
            pred = self.model(X)
            loss = self.loss_fn(pred, y)

            # Backpropagation
            loss.backward()
            self.optimizer.step()
            self.optimizer.zero_grad()

            running_loss += loss.item()*len(X)
            seen += len(X)
            if batch % 50 == 0:
                loss, current = loss.item(), (batch + 1) * len(X)
                tqdm.write(f"\t loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        self.train_loss.append(running_loss/seen)
        self.epoch +=1
        print(f"\t (TRAIN) loss: {running_loss/seen:>7f}")
        return running_loss/seen

    
    @torch.no_grad()
    def validate(self, test_data):
        size = len(test_data.dataset)
        self.model.eval()
        valid_loss, accuracy = 0.0, 0
    
        all_preds, all_labels = [], []
        for X, y in test_data:
            X, y = self._dev_encode(X, y)
    
            logits = self.model(X)                       # logits crus
            valid_loss += self.loss_fn(logits, y).item() * len(X)
            preds = logits.argmax(1)
            accuracy += (preds == y).sum().item()
    
            all_preds.append(preds.cpu())
            all_labels.append(y.cpu())
    
        valid_loss /= size
        accuracy /= size
    
        all_preds  = torch.cat(all_preds).numpy()
        all_labels = torch.cat(all_labels).numpy()
        f1 = f1_score(all_labels, all_preds, average='macro')
    
        self.valid_loss.append(valid_loss)
        self.accuracy.append(accuracy * 100)
        self.f1_score.append(f1)
    
        print(f"\t (VALID) acc: {100*accuracy:>0.1f}%, Avg loss: {valid_loss:>8f}, f1: {f1:.4f}\n")
        return valid_loss, accuracy, f1

    @torch.no_grad()
    def test(self, test_data):
        size = len(test_data.dataset)
        self.model.eval()
        valid_loss, accuracy = 0.0, 0
    
        all_preds, all_labels = [], []
        for X, y in test_data:
            X, y = self._dev_encode(X, y)
    
            logits = self.model(X)                       # logits crus
            valid_loss += self.loss_fn(logits, y).item() * len(X)
            preds = logits.argmax(1)
            accuracy += (preds == y).sum().item()
    
            all_preds.append(preds.cpu())
            all_labels.append(y.cpu())
    
        valid_loss /= size
        accuracy /= size

        all_preds  = torch.cat(all_preds).numpy()
        all_labels = torch.cat(all_labels).numpy()
        f1 = f1_score(all_labels, all_preds, average='macro')
        self.test_stats['accuracy'] = accuracy
        self.test_stats['loss'] = valid_loss
        self.test_stats['f1'] = f1
        print(f"\t (TEST) data acc: {100*accuracy:>0.1f}%, Avg loss: {valid_loss:>8f}, f1: {f1:.4f}\n")
        return valid_loss, accuracy, f1

    
    def _dev_encode(self,*args):
        return tuple(arg.to(self.device) for arg in args)

    def save_checkpoint(self,fname):
        if fname is None:
            return
    
        checkpoint = {
            'epoch': self.epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'validation_loss': self.valid_loss,
            'training_loss': self.train_loss,
            'accuracy':self.accuracy,
            'f1_score':self.f1_score,
            'early_stop_flag':self.early_stop_flag,
            'best_valid_loss': self.best_valid_loss,
            'best_accuracy': self.best_accuracy,
            'best_epoch': self.best_epoch,
            'test_stats': self.test_stats
            }
        torch.save(checkpoint, fname)

    def load_checkpoint(self, fname,boot_from_best=False):
        checkpoint = torch.load(fname, weights_only=False)
        if boot_from_best:
            # print(checkpoint['best_epoch'])
            _slice = slice(checkpoint['best_epoch'])
            current_epoch = checkpoint['best_epoch']
        else:
            _slice = slice(None)
            current_epoch = checkpoint['epoch']

    
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.epoch = current_epoch
        self.valid_loss = checkpoint['validation_loss'][_slice]
        self.train_loss = checkpoint['training_loss'][_slice]
        self.accuracy = checkpoint['accuracy'][_slice]
        self.f1_score = checkpoint['f1_score'][_slice]
        self.early_stop_flag = checkpoint['early_stop_flag']
        self.best_valid_loss = checkpoint['best_valid_loss']
        self.best_accuracy = checkpoint['best_accuracy'] 
        self.best_epoch = checkpoint['best_epoch']
        self.test_stats = checkpoint['test_stats']


## Treino de um fold
Para cada fold, é chamado `evaluate_fold`, reponsável por separar o dataset em validação e teste. Alguns hiperparâmetros foram expostos para otimização com optuna, sendo estes:
- dropout
- learning rate
- dimensões das hidden layers

In [6]:

def evaluate_fold(fold,dropout,lr,transform,train_dl,valid_dl,test_dl,
                  dense1=64,dense2=32,max_epochs=50,patience=7,
                  checkpoint_filename=None):
    
    model = NeuralNetwork(dropout=dropout,dense1=dense1,dense2=dense2)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    trainer = Trainer(model = model, loss_fn = loss_fn,optimizer=optimizer)
    try:
        trainer.load_checkpoint(checkpoint_filename)
        print('checkpoint loaded!')
    except:
        pass
    if trainer.early_stop_flag:
        print('This fold already converged!')
        best_model = trainer.model
        return trainer.best_valid_loss, trainer.best_accuracy, best_model

    best_state = None
    best_acc, early_stop_cnt, best_val_loss, best_f1 = 0, 0, float('inf'),0
    for epoch in range(trainer.epoch,max_epochs):
        print(f':: (FOLD {fold}) Epoch: {epoch} ({epoch-trainer.best_epoch} since best)')
        epoch_t0 = time.time()
        trainer.train(train_dl)
        val_loss, acc,f1 = trainer.validate(valid_dl)
    
        # EarlyStopper
        if acc > trainer.best_accuracy + 1e-3:
            best_f1 = f1
            trainer.best_epoch = epoch
            trainer.best_valid_loss = val_loss
            trainer.best_accuracy = acc
            early_stop_cnt = 0
            best_state = copy.deepcopy(trainer.model.state_dict())
            
            trainer.save_checkpoint(checkpoint_filename)
        else:
            early_stop_cnt += 1
            if early_stop_cnt >= patience:
                trainer.early_stop_flag = True
                break

        epoch_dt = time.time() - epoch_t0
        print(f"\t{epoch_dt/60:.2f} min ({epoch_dt/3600:.2f} h)")
    trainer.model.load_state_dict(best_state)
    trainer.save_checkpoint(checkpoint_filename)
    best_model = trainer.model
    trainer.test(test_dl)
    return trainer.best_valid_loss, trainer.best_accuracy, best_model

## Treinando os folds
- Cada fold roda no máximo 70 épocas, podendo parar antes caso não haja melhora na acurácia.
- Utilizou-se dropout de 0.3 e learning rate de 0.001. Nesse passo, esses valores foram selecionados com base na tentativa e erro para um único fold, buscando minimizar o número de épocas para se obter uma acurácia inferior à 50%.

- reduzi augmentation sets para 2
- Testando apenas para fold 1, epoch 1
    - Feature caching in valid data: off
        - epoch1 acc = 26,9%
    - Feature caching on
        - epoch1 acc = 34,6
     
- lr: 1E-4 

In [9]:
val_losses, val_ac = [], []
dropout = .5
lr = 1E-4
for fold in range(1,11):
    train_dl,valid_dl, test_dl = get_data(fold, fs=fs, transform=transform)
    dataloaders = dict(train_dl=train_dl,valid_dl=valid_dl,test_dl=test_dl)
    vl,vac, _ = evaluate_fold(fold,
                              dropout=dropout, 
                              lr=lr,
                              transform=transform,
                              max_epochs=150, 
                              patience = 15,
                              checkpoint_filename=f'./checkpoints/fold_{fold}.pth',
                              **dataloaders
                             )
    val_losses.append(vl)
    val_ac.append(vac)
    print(f'Fold {fold} trained')

Caching signals:   0%|          | 0/6971 [00:00<?, ?it/s]

Caching features (0/3):   0%|          | 0/6971 [00:00<?, ?it/s]

Caching features (1/3):   0%|          | 0/6971 [00:00<?, ?it/s]

Caching features (2/3):   0%|          | 0/6971 [00:00<?, ?it/s]

Caching features (3/3):   0%|          | 0/6971 [00:00<?, ?it/s]

Caching features (0/0):   0%|          | 0/873 [00:00<?, ?it/s]

... (1,576 outputs left out)

Treino:   0%|          | 0/220 [00:00<?, ?it/s]

	 loss: 0.652349  [   32/ 7022]
	 loss: 0.757799  [ 1632/ 7022]
	 loss: 1.067194  [ 3232/ 7022]
	 loss: 0.615892  [ 4832/ 7022]
	 loss: 0.538528  [ 6432/ 7022]
	 (TRAIN) loss: 0.816659
	 (VALID) acc: 67.0%, Avg loss: 1.027169, f1: 0.6853

	0.44 min (0.01 h)
:: (FOLD 10) Epoch: 63 (15 since best)


Treino:   0%|          | 0/220 [00:00<?, ?it/s]

	 loss: 0.566714  [   32/ 7022]
	 loss: 1.151416  [ 1632/ 7022]
	 loss: 0.947553  [ 3232/ 7022]
	 loss: 0.619051  [ 4832/ 7022]
	 loss: 0.684055  [ 6432/ 7022]
	 (TRAIN) loss: 0.821353
	 (VALID) acc: 69.2%, Avg loss: 1.018138, f1: 0.7067

	 (TEST) data acc: 69.3%, Avg loss: 1.391944, f1: 0.6957

Fold 10 trained


# Otimização de parâmetros
utilizando o optuna para encontrar um bom valor para learning rate, dropout e tamanho das dense layers. Para poupar tempo, 

Note que: o fato de eu ter limitado o número de épocas em 20 favorece modelos que aprendem mais rápido.

~~só deus sabe o que eu fiz com esses prints de loss de acordo com o batch~~

In [7]:
# Caching once, then training multiple times
train,valid,test=[],[],[]
optuna_foldlist = [1,3,5,9]
for fold in optuna_foldlist:
    print(f'caching fold {fold}')
    train_dl,valid_dl, test_dl = get_data(fold, fs=fs, transform=transform)
    train.append(train_dl)
    valid.append(valid_dl)
    test.append(test_dl)

caching fold 1


Caching signals:   0%|          | 0/6971 [00:00<?, ?it/s]

Caching features (0/3):   0%|          | 0/6971 [00:00<?, ?it/s]

Caching features (1/3):   0%|          | 0/6971 [00:00<?, ?it/s]

Caching features (2/3):   0%|          | 0/6971 [00:00<?, ?it/s]

Caching features (3/3):   0%|          | 0/6971 [00:00<?, ?it/s]

... (18 outputs left out)

Caching features (1/3):   0%|          | 0/7043 [00:00<?, ?it/s]

Caching features (2/3):   0%|          | 0/7043 [00:00<?, ?it/s]

Caching features (3/3):   0%|          | 0/7043 [00:00<?, ?it/s]

Caching features (0/0):   0%|          | 0/816 [00:00<?, ?it/s]

In [7]:
print('RUNNING OPTUNA')
def objective(trial):
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    dense1 = trial.suggest_int('dense1',32,128)
    dense2 = trial.suggest_int('dense2',32,dense1)

    val_losses, val_ac = [], []
    for idx,fold in enumerate(optuna_foldlist):
        dataloaders = dict(train_dl=train[idx],valid_dl=valid[idx],test_dl=test[idx])
        vl,vac, _ = evaluate_fold(fold, dropout=dropout, transform=transform,
                                  lr=lr, max_epochs=20, 
                                  patience = 12,
                                  checkpoint_filename=None,
                                  **dataloaders
                                 )
        val_losses.append(vl)
        val_ac.append(vac)
        trial.report(np.mean(val_ac), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(val_ac))

study_name = "OPTUNA_study_lr_dropout_layersize_091126"
storage_name = f"sqlite:///{study_name}.db"
study = optuna.create_study(
    study_name = study_name,
    storage    = storage_name,
    direction  = "maximize",
    sampler    = optuna.samplers.TPESampler(seed=42),
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, 
                                             n_warmup_steps=2),
    load_if_exists = True
)

total_trials = 60
n_trials = total_trials - len(study.trials)
if n_trials >= 0:
    print(f'starting {len(study.trials)}/{total_trials} ({n_trials} remaining)')    
    study.optimize(objective, n_trials=n_trials, timeout=None)

print("Best params:", study.best_params)
print("Best value:", study.best_value)


RUNNING OPTUNA


[I 2026-11-11 22:09:46,209] Using an existing study with `study_name='OPTUNA_study_lr_dropout_layersize_091126'` instead of creating a new one.


Best params: {'dropout': 0.04610870125933331, 'lr': 0.00018487820578222023, 'dense1': 43, 'dense2': 32}
Best value: 0.6673567160305335


In [8]:
from optuna.visualization import plot_contour
from optuna.visualization import plot_edf
from optuna.visualization import plot_intermediate_values
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_rank
from optuna.visualization import plot_slice
from optuna.visualization import plot_timeline

In [9]:
plot_optimization_history(study)

In [12]:
# plot_contour(study)
# plot_parallel_coordinate(study)


In [15]:
plot_param_importances(study)

/tmp/ipykernel_49/3836449081.py:1: UserWarning:

Some regimes for parameter `dense2` have less than 2 trials. The importance of the parameter may be inaccurate.



# Rodando com o novo conjunto de parâmetros

In [ ]:

val_losses, val_ac = [], []
dropout = 0.04
lr = 0.00018
dense1 = 43
dense2 = 32
for fold in range(1,11):
    train_dl,valid_dl, test_dl = get_data(fold, fs=fs, transform=transform)
    dataloaders = dict(train_dl=train_dl,valid_dl=valid_dl,test_dl=test_dl)
    vl,vac, _ = evaluate_fold(fold,
                              dropout=dropout, 
                              lr=lr,
                              transform=transform,
                              max_epochs=150, 
                              patience = 15,
                              dense1=dense1,dense2=dense2,
                              checkpoint_filename=f'./checkpoints/_optim_fold_{fold}.pth',
                              **dataloaders
                             )
    val_losses.append(vl)
    val_ac.append(vac)
    print(f'Fold {fold} trained')

Caching signals:   0%|          | 0/6971 [00:00<?, ?it/s]

Caching features (0/3):   0%|          | 0/6971 [00:00<?, ?it/s]

Caching features (1/3):   0%|          | 0/6971 [00:00<?, ?it/s]

Caching features (2/3):   0%|          | 0/6971 [00:00<?, ?it/s]

Caching features (3/3):   0%|          | 0/6971 [00:00<?, ?it/s]

Caching features (0/0):   0%|          | 0/873 [00:00<?, ?it/s]

... (518 outputs left out)

	 loss: 0.054412  [   32/ 6923]
	 loss: 0.170881  [ 1632/ 6923]
	 loss: 0.071857  [ 3232/ 6923]
	 loss: 0.117312  [ 4832/ 6923]
	 loss: 0.911779  [ 6432/ 6923]
	 (TRAIN) loss: 0.186191
	 (VALID) acc: 71.7%, Avg loss: 1.245618, f1: 0.7247

	 (TEST) data acc: 56.8%, Avg loss: 2.240278, f1: 0.5898

Fold 5 trained


Caching signals:   0%|          | 0/7036 [00:00<?, ?it/s]

Caching features (0/3):   0%|          | 0/7036 [00:00<?, ?it/s]

Caching features (1/3):   0%|          | 0/7036 [00:00<?, ?it/s]